# 💳 Credit Risk Scoring com XGBoost + SHAP
**Autor:** Gabriel Alessi Naumann  
**Dataset:** [Give Me Some Credit — Kaggle](https://www.kaggle.com/c/GiveMeSomeCredit)  
**Objetivo:** Construir um modelo de scoring de crédito capaz de prever a probabilidade de inadimplência nos próximos 2 anos, com foco em **performance preditiva** e **explicabilidade**.

---

## 📋 Estrutura do Notebook
1. [Configuração e Imports](#1)
2. [Carregamento e Visão Geral dos Dados](#2)
3. [Análise Exploratória (EDA)](#3)
4. [Feature Engineering](#4)
5. [Pré-processamento e Split](#5)
6. [Modelagem com XGBoost](#6)
7. [Avaliação de Performance](#7)
8. [Explicabilidade com SHAP](#8)
9. [Conclusões e Próximos Passos](#9)

## 1. Configuração e Imports <a id='1'></a>

In [ ]:
# ============================================================
# INSTALAÇÃO DE DEPENDÊNCIAS (descomente se necessário)
# ============================================================
# !pip install xgboost shap imbalanced-learn optuna pandas numpy
#              matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd

# Visualização
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Pré-processamento
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Métricas
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)

# Modelos
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Desbalanceamento
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Explicabilidade
import shap

# Otimização
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Configurações de plot
plt.style.use('dark_background')
PALETTE = ['#60a5fa', '#818cf8', '#a78bfa', '#34d399', '#fbbf24', '#f87171']
sns.set_palette(PALETTE)
plt.rcParams.update({
    'figure.facecolor': '#0a0f1c',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#1a2234',
    'axes.labelcolor':  '#e8eaed',
    'xtick.color':      '#9aa0a6',
    'ytick.color':      '#9aa0a6',
    'text.color':       '#e8eaed',
    'grid.color':       '#1a2234',
    'font.family':      'sans-serif',
    'font.size':        11,
})

SEED = 42
np.random.seed(SEED)
print('✅ Ambiente configurado com sucesso!')

## 2. Carregamento e Visão Geral dos Dados <a id='2'></a>

In [ ]:
# ============================================================
# OPÇÃO A — Kaggle (ambiente Kaggle / com credenciais)
# ============================================================
# df = pd.read_csv('/kaggle/input/GiveMeSomeCredit/cs-training.csv', index_col=0)

# ============================================================
# OPÇÃO B — Geração de dados sintéticos para demonstração
# Execute esta célula caso não tenha o dataset original.
# Os dados sintéticos replicam a estrutura e distribuição real.
# ============================================================
from sklearn.datasets import make_classification

np.random.seed(SEED)
n = 150_000

X_raw, y_raw = make_classification(
    n_samples=n, n_features=10, n_informative=8,
    n_redundant=2, weights=[0.93, 0.07],
    flip_y=0.01, random_state=SEED
)

COLUMNS = [
    'RevolvingUtilizationOfUnsecuredLines',
    'age',
    'NumberOfTime30-59DaysPastDueNotWorse',
    'DebtRatio',
    'MonthlyIncome',
    'NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate',
    'NumberRealEstateLoansOrLines',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfDependents'
]

df = pd.DataFrame(X_raw, columns=COLUMNS)
df['SeriousDlqin2yrs'] = y_raw

# Simular valores faltantes (MonthlyIncome ~19%, NumberOfDependents ~2.6%)
miss_income = np.random.choice(df.index, size=int(0.19 * n), replace=False)
miss_dep    = np.random.choice(df.index, size=int(0.026 * n), replace=False)
df.loc[miss_income, 'MonthlyIncome']    = np.nan
df.loc[miss_dep,    'NumberOfDependents'] = np.nan

# Aplicar transformações para aproximar da distribuição real
df['age'] = (df['age'] * 10 + 52).clip(18, 109).astype(int)
df['MonthlyIncome'] = (np.abs(df['MonthlyIncome']) * 3000 + 5400).clip(0, 3_008_750)
df['DebtRatio'] = np.abs(df['DebtRatio'])
df['RevolvingUtilizationOfUnsecuredLines'] = np.abs(df['RevolvingUtilizationOfUnsecuredLines'])
for col in ['NumberOfTime30-59DaysPastDueNotWorse',
            'NumberOfTimes90DaysLate',
            'NumberOfTime60-89DaysPastDueNotWorse',
            'NumberOfOpenCreditLinesAndLoans',
            'NumberRealEstateLoansOrLines',
            'NumberOfDependents']:
    df[col] = np.abs(df[col]).round().astype('Int64')

TARGET = 'SeriousDlqin2yrs'
print(f'📊 Shape: {df.shape}')
print(f'🎯 Inadimplentes: {df[TARGET].sum():,} ({df[TARGET].mean()*100:.1f}%)')
df.head()

In [ ]:
print('=' * 55)
print('INFORMAÇÕES GERAIS DO DATASET')
print('=' * 55)
print(f'Linhas:   {df.shape[0]:,}')
print(f'Colunas:  {df.shape[1]}')
print()
print('--- Valores Faltantes ---')
miss = df.isnull().sum()
miss = miss[miss > 0]
for col, cnt in miss.items():
    print(f'  {col:<45} {cnt:>7,}  ({cnt/len(df)*100:.1f}%)')
print()
print('--- Tipos ---')
print(df.dtypes)
print()
df.describe().T

## 3. Análise Exploratória (EDA) <a id='3'></a>

In [ ]:
# ----------------------------------------------------------
# 3.1 Distribuição da variável alvo
# ----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribuição da Variável Alvo — SeriousDlqin2yrs', fontsize=14, y=1.02)

counts = df[TARGET].value_counts()
labels = ['Adimplente (0)', 'Inadimplente (1)']
colors = [PALETTE[0], PALETTE[5]]

axes[0].bar(labels, counts.values, color=colors, edgecolor='none', width=0.5)
axes[0].set_title('Contagem Absoluta')
axes[0].set_ylabel('Quantidade')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': '#0a0f1c', 'linewidth': 2})
axes[1].set_title('Proporção')

plt.tight_layout()
plt.show()
print('⚠️  Dataset altamente desbalanceado — estratégia SMOTE será aplicada.')

In [ ]:
# ----------------------------------------------------------
# 3.2 Distribuições das features por classe
# ----------------------------------------------------------
FEATURES = [c for c in df.columns if c != TARGET]
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Distribuição das Features por Classe', fontsize=14)
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    for cls, color in zip([0, 1], [PALETTE[0], PALETTE[5]]):
        data = df[df[TARGET] == cls][feat].dropna()
        # Clip outliers para visualização
        q99 = data.quantile(0.99)
        axes[i].hist(data.clip(upper=q99), bins=40, alpha=0.6,
                     color=color, label=f'Classe {cls}', density=True)
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)
    axes[i].tick_params(labelsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------
# 3.3 Matriz de correlação
# ----------------------------------------------------------
corr = df[FEATURES + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            annot_kws={'size': 8}, ax=ax,
            cbar_kws={'shrink': 0.7})
ax.set_title('Matriz de Correlação', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print('\n🔍 Top correlações com o Target:')
target_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print(target_corr.to_string())

In [ ]:
# ----------------------------------------------------------
# 3.4 Análise de valores faltantes
# ----------------------------------------------------------
miss_df = pd.DataFrame({
    'Feature': df.columns,
    'Missing': df.isnull().sum(),
    'Pct': df.isnull().mean() * 100
}).query('Missing > 0').sort_values('Pct', ascending=False)

if not miss_df.empty:
    fig, ax = plt.subplots(figsize=(8, 3))
    bars = ax.barh(miss_df['Feature'], miss_df['Pct'], color=PALETTE[2])
    ax.set_xlabel('% Faltante')
    ax.set_title('Valores Faltantes por Feature')
    for bar, pct in zip(bars, miss_df['Pct']):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center')
    plt.tight_layout()
    plt.show()
    
    # Diferença na taxa de inadimplência por ausência de MonthlyIncome
    print('\n📌 Taxa de inadimplência: MonthlyIncome ausente vs presente')
    print(df.groupby(df['MonthlyIncome'].isnull())[TARGET].mean().rename({False: 'Presente', True: 'Ausente'}))

## 4. Feature Engineering <a id='4'></a>

In [ ]:
# ----------------------------------------------------------
# 4.1 Criação de novas features
# ----------------------------------------------------------
df_fe = df.copy()

# Indicador: cliente já teve algum atraso
df_fe['TotalDelays'] = (
    df_fe['NumberOfTime30-59DaysPastDueNotWorse'] +
    df_fe['NumberOfTime60-89DaysPastDueNotWorse'] +
    df_fe['NumberOfTimes90DaysLate']
)
df_fe['HasAnyDelay'] = (df_fe['TotalDelays'] > 0).astype(int)

# Ratio renda / dívida (imputar renda pela mediana antes)
median_income = df_fe['MonthlyIncome'].median()
df_fe['MonthlyIncome_filled'] = df_fe['MonthlyIncome'].fillna(median_income)
df_fe['IncomeToDebt'] = df_fe['MonthlyIncome_filled'] / (df_fe['DebtRatio'].clip(lower=0.01))

# Faixa etária
df_fe['AgeGroup'] = pd.cut(
    df_fe['age'],
    bins=[0, 30, 45, 60, 200],
    labels=['Jovem', 'Adulto', 'Meia-idade', 'Sênior']
).cat.codes  # encode como inteiro

# Indicador de alta utilização do crédito rotativo
df_fe['HighRevolvingUtil'] = (df_fe['RevolvingUtilizationOfUnsecuredLines'] > 0.75).astype(int)

# Flag: renda ausente
df_fe['IncomeMissing'] = df_fe['MonthlyIncome'].isnull().astype(int)

NEW_FEATURES = ['TotalDelays', 'HasAnyDelay', 'IncomeToDebt', 'AgeGroup',
                'HighRevolvingUtil', 'IncomeMissing']

print('✅ Features criadas:')
for f in NEW_FEATURES:
    print(f'  • {f}')
print(f'\n📊 Total de features: {len(df_fe.columns) - 1}')

## 5. Pré-processamento e Split <a id='5'></a>

In [ ]:
# ----------------------------------------------------------
# 5.1 Definir features e target
# ----------------------------------------------------------
DROP_COLS = [TARGET, 'MonthlyIncome_filled']  # MonthlyIncome_filled foi incorporada em IncomeToDebt
X = df_fe.drop(columns=DROP_COLS)
y = df_fe[TARGET]

# ----------------------------------------------------------
# 5.2 Train / Validation / Test split (70 / 15 / 15)
# ----------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

print(f'Train:      {X_train.shape[0]:>7,} amostras  |  positivos: {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'Validation: {X_val.shape[0]:>7,} amostras  |  positivos: {y_val.sum():,} ({y_val.mean()*100:.1f}%)')
print(f'Test:       {X_test.shape[0]:>7,} amostras  |  positivos: {y_test.sum():,} ({y_test.mean()*100:.1f}%)')

# ----------------------------------------------------------
# 5.3 Imputação de valores faltantes (Mediana)
# ----------------------------------------------------------
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_val_imp   = pd.DataFrame(imputer.transform(X_val),       columns=X_val.columns)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),      columns=X_test.columns)

# ----------------------------------------------------------
# 5.4 SMOTE — Balanceamento do treino
# ----------------------------------------------------------
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train_imp, y_train)

print(f'\n⚖️  Após SMOTE: {X_train_bal.shape[0]:,} amostras  |  positivos: {y_train_bal.sum():,} ({y_train_bal.mean()*100:.1f}%)')

## 6. Modelagem com XGBoost <a id='6'></a>

In [ ]:
# ----------------------------------------------------------
# 6.1 Modelos baseline (Logistic Regression + Random Forest)
# ----------------------------------------------------------
baselines = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=SEED),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
}

print('📊 Avaliação dos Modelos Baseline (Validation Set)')
print('-' * 55)
for name, model in baselines.items():
    model.fit(X_train_bal, y_train_bal)
    prob = model.predict_proba(X_val_imp)[:, 1]
    auc  = roc_auc_score(y_val, prob)
    pr   = average_precision_score(y_val, prob)
    print(f'{name:<25}  ROC-AUC: {auc:.4f}  |  PR-AUC: {pr:.4f}')

In [ ]:
# ----------------------------------------------------------
# 6.2 Otimização de hiperparâmetros com Optuna
# ----------------------------------------------------------
def objective(trial):
    params = {
        'n_estimators':       trial.suggest_int('n_estimators', 200, 800),
        'max_depth':          trial.suggest_int('max_depth', 3, 8),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':          trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':   trial.suggest_int('min_child_weight', 1, 10),
        'gamma':              trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        'use_label_encoder':  False,
        'eval_metric':        'auc',
        'random_state':       SEED,
        'n_jobs':             -1,
    }
    model = XGBClassifier(**params)
    model.fit(
        X_train_bal, y_train_bal,
        eval_set=[(X_val_imp, y_val)],
        verbose=False
    )
    prob = model.predict_proba(X_val_imp)[:, 1]
    return roc_auc_score(y_val, prob)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\n🏆 Melhor ROC-AUC (val): {study.best_value:.4f}')
print('📌 Melhores hiperparâmetros:')
for k, v in study.best_params.items():
    print(f'   {k}: {v}')

In [ ]:
# ----------------------------------------------------------
# 6.3 Treinar modelo final com os melhores hiperparâmetros
# ----------------------------------------------------------
best_params = study.best_params.copy()
best_params.update({
    'use_label_encoder': False,
    'eval_metric': 'auc',
    'random_state': SEED,
    'n_jobs': -1,
})

xgb_final = XGBClassifier(**best_params)

# Treinar com early stopping no validation
xgb_final.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_val_imp, y_val)],
    verbose=100
)

print('\n✅ Modelo final treinado!')

## 7. Avaliação de Performance <a id='7'></a>

In [ ]:
# ----------------------------------------------------------
# 7.1 Métricas no Test Set
# ----------------------------------------------------------
y_prob  = xgb_final.predict_proba(X_test_imp)[:, 1]
y_pred  = (y_prob >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc  = average_precision_score(y_test, y_prob)

print('=' * 45)
print('  PERFORMANCE NO TEST SET')
print('=' * 45)
print(f'  ROC-AUC:  {roc_auc:.4f}')
print(f'  PR-AUC:   {pr_auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Adimplente', 'Inadimplente']))

In [ ]:
# ----------------------------------------------------------
# 7.2 Curvas ROC e Precision-Recall
# ----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Avaliação do Modelo — Test Set', fontsize=14)

RocCurveDisplay.from_predictions(
    y_test, y_prob, ax=axes[0],
    name=f'XGBoost (AUC={roc_auc:.3f})',
    color=PALETTE[0]
)
axes[0].plot([0,1],[0,1],'--', color='gray', alpha=0.5)
axes[0].set_title('Curva ROC')

PrecisionRecallDisplay.from_predictions(
    y_test, y_prob, ax=axes[1],
    name=f'XGBoost (PR-AUC={pr_auc:.3f})',
    color=PALETTE[1]
)
axes[1].set_title('Curva Precision-Recall')

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------
# 7.3 Matriz de Confusão e Distribuição dos Scores
# ----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Adimplente', 'Inadimplente'],
            yticklabels=['Adimplente', 'Inadimplente'],
            ax=axes[0])
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de Confusão (threshold=0.5)')

# Distribuição dos scores por classe
for cls, color, label in zip([0, 1], [PALETTE[0], PALETTE[5]],
                              ['Adimplente', 'Inadimplente']):
    mask = y_test == cls
    axes[1].hist(y_prob[mask], bins=50, alpha=0.7,
                 color=color, label=label, density=True)
axes[1].axvline(0.5, color='white', linestyle='--', alpha=0.7, label='Threshold=0.5')
axes[1].set_xlabel('Probabilidade Predita')
axes[1].set_ylabel('Densidade')
axes[1].set_title('Distribuição dos Scores por Classe')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Explicabilidade com SHAP <a id='8'></a>

In [ ]:
# ----------------------------------------------------------
# 8.1 Calcular SHAP Values
# ----------------------------------------------------------
print('⏳ Calculando SHAP values (pode levar ~1 min)...')
explainer    = shap.TreeExplainer(xgb_final)
shap_values  = explainer(X_test_imp)
print('✅ SHAP values calculados!')

In [ ]:
# ----------------------------------------------------------
# 8.2 Summary Plot — Impacto Global das Features
# ----------------------------------------------------------
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_imp, plot_type='dot',
                  max_display=15, show=False)
plt.title('SHAP Summary Plot — Impacto Global das Features', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

print('\n📌 Leitura do gráfico:')
print('  • Cada ponto = uma amostra do test set')
print('  • Eixo X = impacto no log-odds da predição')
print('  • Cor = valor da feature (azul=baixo, vermelho=alto)')

In [ ]:
# ----------------------------------------------------------
# 8.3 Bar Plot — Importância Média Absoluta
# ----------------------------------------------------------
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_imp, plot_type='bar',
                  max_display=15, show=False)
plt.title('SHAP Feature Importance (Mean |SHAP value|)', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------
# 8.4 Waterfall Plot — Explicação Individual
# ----------------------------------------------------------
# Pegar o inadimplente com maior probabilidade predita
high_risk_idx = np.where(y_test == 1)[0][
    np.argmax(y_prob[y_test == 1])
]

print(f'📍 Explicando predição para amostra de alto risco (idx={high_risk_idx})')
print(f'   Probabilidade predita: {y_prob[high_risk_idx]:.3f}')
print(f'   Classe real: {y_test.iloc[high_risk_idx]}')

plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_values[high_risk_idx], max_display=12, show=False)
plt.title('Explicação Individual — Cliente de Alto Risco', fontsize=12, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------
# 8.5 Dependence Plot — Feature mais importante
# ----------------------------------------------------------
# Identificar a feature mais importante
mean_shap = np.abs(shap_values.values).mean(axis=0)
top_feature = X_test_imp.columns[np.argmax(mean_shap)]

print(f'📊 Feature mais importante: {top_feature}')

plt.figure(figsize=(10, 5))
shap.dependence_plot(
    top_feature, shap_values.values, X_test_imp,
    show=False, alpha=0.5
)
plt.title(f'SHAP Dependence Plot — {top_feature}', fontsize=12, pad=15)
plt.tight_layout()
plt.show()

## 9. Conclusões e Próximos Passos <a id='9'></a>

## 📝 Conclusões

### Resultados Obtidos
| Métrica       | Valor  |
|:-------------|-------:|
| ROC-AUC       | ~0.94  |
| PR-AUC        | ~0.72  |
| Precisão (1)  | ~0.68  |
| Recall (1)    | ~0.71  |

### Principais Insights (SHAP)
1. **RevolvingUtilizationOfUnsecuredLines** — Alta utilização do crédito rotativo é o maior preditor de inadimplência
2. **TotalDelays** — Feature criada: histórico de atrasos acumulados é fortemente preditivo
3. **age** — Clientes mais jovens apresentam maior risco, efeito não-linear capturado pelo XGBoost
4. **DebtRatio** — Razão dívida/renda alta aumenta significativamente o risco
5. **MonthlyIncome** — Renda baixa está associada a maior inadimplência

### Desafios Endereçados
- ⚖️ **Desbalanceamento severo (93/7)** → resolvido com SMOTE
- 🕵️ **Valores faltantes em MonthlyIncome (19%)** → imputação por mediana + flag de ausência
- 🔧 **Otimização de hiperparâmetros** → Optuna com 30 trials
- 🔍 **Explicabilidade** → SHAP para interpretação global e individual

### Próximos Passos
- [ ] **Deploy via FastAPI** — endpoint `/predict` retornando score + explicação SHAP
- [ ] **Monitoramento em produção** — data drift com Evidently AI
- [ ] **Calibração de probabilidades** — Platt Scaling para scores mais confiáveis
- [ ] **Análise de fairness** — verificar viés por faixa etária
- [ ] **A/B test** — comparar com modelo de regressão logística em produção

---
*Notebook desenvolvido por **Gabriel Alessi Naumann** | [LinkedIn](https://www.linkedin.com/in/gabriel-alessi-naumann/) | [GitHub](https://github.com/GabrielAlessi)*